In [1]:
from dotenv import load_dotenv
load_dotenv()

import os

import numpy as np
from pixell import enmap, enplot, reproject
import glob
import matplotlib.pyplot as plt
import emcee, corner
from astropy.io import fits
import sys
from astropy import units as u, constants as const
sys.path.insert(0, '../src')
#sys.path.insert(0, "/home/gill/apps/szpack/python")
import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
import yaml
import itertools
from pixell import enmap

# Compton y calculation
from astropy.constants import k_B, m_p, c, sigma_T, m_e
from astropy import units as u

from astropy.coordinates import SkyCoord
import astropy.units as u

import utils as ut


# latex font
plt.rc('text', usetex=True)
plt.rc('font', family='sans-serif', size=22)

In [2]:
def calc_y_and_err(tau, tau_err, Te, Te_err):
    Te = Te * u.keV
    Te_err = Te_err * u.keV
    Te_K = (Te / k_B).to(u.K)    
    Te_err_K = (Te_err / k_B).to(u.K)

    y = tau * k_B * Te_K / (m_e * c**2).to(u.J)
    yerr = y * np.sqrt((tau_err / tau)**2 + (Te_err_K / Te_K)**2)

    return y.value, yerr.value

def get_y_vector(samples, idx_tau, idx_Te):
    Te = samples[:, idx_Te] * u.keV
    tau = samples[:, idx_tau]
    Te_K = (Te / k_B).to(u.K)
    y = tau * k_B * Te_K / (m_e * c**2).to(u.J)
    return y.value

def calc_mean_and_std(samples, idx):
    data = samples[:, idx]

    mcmc_run = np.percentile(data, [16, 50, 84])
    err = .5 * (mcmc_run[2] - mcmc_run[0])
    
    # Calculate bin width using Freedman-Diaconis rule
    n = len(data)
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    bin_width = 2 * iqr * n**(-1/3)

    # Calculate number of bins using bin width
    num_bins = int((np.max(data) - np.min(data)) / bin_width)

    # Create histogram
    hist, bin_edges = np.histogram(data, bins=num_bins, density=True)

    # Find bin with maximum value (estimate of mode)
    mode_bin = np.argmax(hist)
    mode_value = 0.5 * (bin_edges[mode_bin] + bin_edges[mode_bin + 1])

    mean = mode_value
    std = err

    #mean = np.mean(data)
    #mean = mcmc_run[1]
    return mean, std

In [3]:
# indices for the chain parameters
param_indices = {
    'ra_a401': 0,
    'dec_a401': 1,
    'beta_a401': 2,
    'rc_a401': 3,
    'R_a401': 4,
    'theta_a401': 5,
    'tau_a401': 6,
    'Te_a401': 7,
    'AD_a401': 8,
    'vr_a401': 9,

    'ra_a399': 10,
    'dec_a399': 11,
    'beta_a399': 12,
    'rc_a399': 13,
    'R_a399': 14,
    'theta_a399': 15,
    'tau_a399': 16,
    'Te_a399': 17,
    'AD_a399': 18,
    'vr_a399': 19,

    'fil_ra': 20,
    'fil_dec': 21,
    'fil_l0': 22,
    'fil_w0': 23,
    'tau_fil': 24,
    'Te_fil': 25,
    'fil_AD': 26,
    'fil_vavg': 27
}

case23 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case23/chain.h5"
case34 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case34/chain.h5"
case35 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case35/chain.h5" 
case36 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case36/chain.h5"
case37 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case37/chain.h5"
case38 = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/case38/chain.h5"

In [4]:
burnin = 50000
thin = 1
samples_case23 = emcee.backends.HDFBackend(case23).get_chain(discard=60000, flat=True, thin=thin)
samples_case34 = emcee.backends.HDFBackend(case34).get_chain(discard=30000, flat=True, thin=thin)
samples_case35 = emcee.backends.HDFBackend(case35).get_chain(discard=30000, flat=True, thin=thin)
samples_case36 = emcee.backends.HDFBackend(case36).get_chain(discard=30000, flat=True, thin=thin)
samples_case37 = emcee.backends.HDFBackend(case37).get_chain(discard=30000, flat=True, thin=thin)
samples_case38 = emcee.backends.HDFBackend(case38).get_chain(discard=30000, flat=True, thin=thin)

print("Number of samples for case 23: {:.0f}".format(samples_case23.shape[0] / emcee.backends.HDFBackend(case23).shape[0]))
print("Number of iterations for case 34: {:.0f}".format(samples_case34.shape[0] / emcee.backends.HDFBackend(case34).shape[0]))
print("Number of iterations for case 35: {:.0f}".format(samples_case35.shape[0] / emcee.backends.HDFBackend(case35).shape[0]))
print("Number of iterations for case 36: {:.0f}".format(samples_case36.shape[0] / emcee.backends.HDFBackend(case36).shape[0]))
print("Number of iterations for case 37: {:.0f}".format(samples_case37.shape[0] / emcee.backends.HDFBackend(case37).shape[0]))
print("Number of iterations for case 38: {:.0f}".format(samples_case38.shape[0] / emcee.backends.HDFBackend(case38).shape[0]))

Number of samples for case 23: 65830
Number of iterations for case 34: 10472
Number of iterations for case 35: 10396
Number of iterations for case 36: 9750
Number of iterations for case 37: 9721
Number of iterations for case 38: 55509


In [5]:
cases = [
    (samples_case23, "case23"),
    (samples_case34, "case34"),
    (samples_case35, "case35"),
    (samples_case36, "case36"),
    (samples_case37, "case37"),
    (samples_case38, "case38")
]

case_labels = ["Temperature: baseline (X-ray)", 
               r"5\% higher than X-ray", 
               r"10\% higher than X-ray", r"15\% higher than X-ray", 
               r"20\% higher than X-ray", r"25\% higher than X-ray"]

In [6]:
# Compton y fits
# --- A401 ---
# Compton-y fit
compton_y_fits = {
    "ra_a401_mean": 44.751,
    "ra_a401_std": 0.002,
    "dec_a401_mean": 13.572,
    "dec_a401_std": 0.002,
    "beta_a401_mean": 0.82,
    "beta_a401_std": 0.05,
    "rc_a401_mean": 2.6,
    "rc_a401_std": 0.35,
    "R_a401_mean": 0.82,
    "R_a401_std": 0.055,
    "theta_a401_mean": 123,
    "theta_a401_std": 8.5,
    "y_a401_mean": 1.260e-04,
    "y_a401_std": 6.00e-06,
    
    "ra_a399_mean": 44.473,
    "ra_a399_std": 0.004,
    "dec_a399_mean": 13.03,
    "dec_a399_std": 0.003,
    "beta_a399_mean": 0.81,
    "beta_a399_std": 0.095,
    "rc_a399_mean": 3.0,
    "rc_a399_std": 0.65,
    "R_a399_mean": 0.93,
    "R_a399_std": 0.06,
    "theta_a399_mean": 133,
    "theta_a399_std": 26.5,
    "y_a399_mean": 8.100e-05,
    "y_a399_std": 6.00e-06,
    
    "fil_ra_mean": 44.68,
    "fil_ra_std": 0.02,
    "fil_dec_mean": 13.37,
    "fil_dec_std": 0.025,
    "fil_l0_mean": 12.3,
    "fil_l0_std": 1.55,
    "fil_w0_mean": 10.8,
    "fil_w0_std": 1.05,
    "y_fil_mean": 1.100e-05,
    "y_fil_std": 1.750e-06,
}

indv_all_fits = {
    # Abell 401 values from "Multi-frequency (individual velocities)" column
    "ra_a401_mean": 44.741,
    "ra_a401_std": 0.002,
    "dec_a401_mean": 13.580,
    "dec_a401_std": 0.002,
    "beta_a401_mean": 1.254,
    "beta_a401_std": 0.091,
    "rc_a401_mean": 4.971,
    "rc_a401_std": 0.525,
    "R_a401_mean": 0.795,
    "R_a401_std": 0.055,
    "theta_a401_mean": 107.4,
    "theta_a401_std": 6.6,
    "y_a401_mean": 1.176e-04,
    "y_a401_std": 8.6e-06,
    "Te_a401_mean": 8.414,
    "Te_a401_std": 0.248,
    "tau_a401_mean": 7.145e-03,
    "tau_a401_std": 0.480e-03,
    "a401_AD_mean": 8.08e4,
    "a401_AD_std": 1.99e5,

    # Abell 399 values from "Multi-frequency (individual velocities)" column
    "ra_a399_mean": 44.465,
    "ra_a399_std": 0.003,
    "dec_a399_mean": 13.040,
    "dec_a399_std": 0.004,
    "beta_a399_mean": 1.000,
    "beta_a399_std": 0.069,
    "rc_a399_mean": 3.998,
    "rc_a399_std": 0.539,
    "R_a399_mean": 0.884,
    "R_a399_std": 0.091,
    "theta_a399_mean": 107.330,
    "theta_a399_std": 27.204,
    "y_a399_mean": 9.114e-05,
    "y_a399_std": 7.01e-06,
    "Te_a399_mean": 7.240,
    "Te_a399_std": 0.186,
    "tau_a399_mean": 6.432e-03,
    "tau_a399_std": 0.466e-03,
    "a399_AD_mean": 1097,
    "a399_AD_std": 1.48e5,

    # Filament values from "Multi-frequency (individual velocities)" column
    "fil_ra_mean": 44.670,
    "fil_ra_std": 0.021,
    "fil_dec_mean": 13.338,
    "fil_dec_std": 0.031,
    "fil_l0_mean": 15.191,
    "fil_l0_std": 2.069,
    "fil_w0_mean": 12.310,
    "fil_w0_std": 1.127,
    "y_fil_mean": 1.450e-05,
    "y_fil_std": 2.16e-06,
    "Te_fil_mean": 6.537,
    "Te_fil_std": 0.350,
    "tau_fil_mean": 1.133e-03,
    "tau_fil_std": 0.158e-03,
    "fil_AD_mean": 3932.99,
    "fil_AD_std": 7.66e4,

    # Velocity parameters for individual fits
    "vr_a401_mean": -1152.99,
    "vr_a401_std": 712.04,
    "vr_a399_mean": 780.43,
    "vr_a399_std": 683.57,
    "fil_vavg_mean": 873.47,
    "fil_vavg_std": 1559.04,
}

In [ ]:
# def result_comparer(samples, compton_y_fits, param_indices, cf_path):
#     # create a dictionary of the sample results
#     # create the labels first for the dictionary

#     ## Take care of RA and DEC values 
#     cf = ut.get_config_file(f'{cf_path}')

#     region = ut.get_region(cf['region_center_ra'], 
#                            cf['region_center_dec'], 
#                            cf['region_width'])

#     dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"
#     data_ref = enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
#                               box=region)
    
#     a401_ra_pix = samples[:, param_indices['ra_a401']]
#     a401_dec_pix = samples[:, param_indices['dec_a401']]

#     a399_ra_pix = samples[:, param_indices['ra_a399']]
#     a399_dec_pix = samples[:, param_indices['dec_a399']]

#     fil_ra_pix = samples[:, param_indices['fil_ra']]
#     fil_dec_pix = samples[:, param_indices['fil_dec']]

#     ra_a401, dec_a401 = data_ref.wcs.celestial.wcs_pix2world(a401_ra_pix, a401_dec_pix, 0)
#     samples[:, param_indices['ra_a401']] = ra_a401
#     samples[:, param_indices['dec_a401']] = dec_a401

#     ra_a399, dec_a399 = data_ref.wcs.celestial.wcs_pix2world(a399_ra_pix, a399_dec_pix, 0)
#     samples[:, param_indices['ra_a399']] = ra_a399
#     samples[:, param_indices['dec_a399']] = dec_a399

#     fil_ra, fil_dec = data_ref.wcs.celestial.wcs_pix2world(fil_ra_pix, fil_dec_pix, 0)
#     samples[:, param_indices['fil_ra']] = fil_ra
#     samples[:, param_indices['fil_dec']] = fil_dec

#     samples[:, param_indices["fil_l0"]] *= 0.5
#     samples[:, param_indices["fil_w0"]] *= 0.5

#     samples[:, param_indices["R_a401"]] = 1 / samples[:, param_indices["R_a401"]]
#     samples[:, param_indices["R_a399"]] = 1 / samples[:, param_indices["R_a399"]]
    
#     # create a dictionary to hold the results
#     labels = {
#         "ra_a401_mean": 0,
#         "ra_a401_std": 0,

#         "dec_a401_mean": 0,
#         "dec_a401_std": 0,

#         "beta_a401_mean": 0,
#         "beta_a401_std": 0,

#         "rc_a401_mean": 0,
#         "rc_a401_std": 0,

#         "R_a401_mean": 0,
#         "R_a401_std": 0,

#         "theta_a401_mean": 0,
#         "theta_a401_std": 0,

#         "tau_a401_mean": 0,
#         "tau_a401_std": 0,

#         "Te_a401_mean": 0,
#         "Te_a401_std": 0,

#         "AD_a401_mean": 0,
#         "AD_a401_std": 0,

#         "vr_a401_mean": 0,
#         "vr_a401_std": 0,

#         "ra_a399_mean": 0,
#         "ra_a399_std": 0,

#         "dec_a399_mean": 0,
#         "dec_a399_std": 0,

#         "beta_a399_mean": 0,
#         "beta_a399_std": 0,

#         "rc_a399_mean": 0,
#         "rc_a399_std": 0,

#         "R_a399_mean": 0,
#         "R_a399_std": 0,

#         "theta_a399_mean": 0,
#         "theta_a399_std": 0,

#         "tau_a399_mean": 0,
#         "tau_a399_std": 0,

#         "Te_a399_mean": 0,
#         "Te_a399_std": 0,

#         "AD_a399_mean": 0,
#         "AD_a399_std": 0,

#         "vr_a399_mean": 0,
#         "vr_a399_std": 0,

#         "fil_ra_mean": 0,
#         "fil_ra_std": 0,

#         "fil_dec_mean": 0,
#         "fil_dec_std": 0,

#         "fil_l0_mean": 0,
#         "fil_l0_std": 0,

#         "fil_w0_mean": 0,
#         "fil_w0_std": 0,

#         "tau_fil_mean": 0,
#         "tau_fil_std": 0,

#         "Te_fil_mean": 0,
#         "Te_fil_std": 0,

#         "fil_AD_mean": 0,
#         "fil_AD_std": 0,

#         "fil_vavg_mean": 0,
#         "fil_vavg_std": 0,

#         "y_a401_mean": 0,
#         "y_a401_std": 0,
        
#         "y_a399_mean": 0,
#         "y_a399_std": 0,

#         "y_fil_mean": 0,
#         "y_fil_std": 0,
#     }

#     # calculate the mean and std for each param in labels
#     for key in labels.keys():
#         if "std" in key:
#             continue
        
#         if "y_" in key:
#             object_name = key.split('_')[1]
#             Te = labels[f'Te_{object_name}_mean']
#             Te_err = labels[f'Te_{object_name}_std']
#             tau = labels[f'tau_{object_name}_mean']
#             tau_err = labels[f'tau_{object_name}_std']
#             y, yerr = calc_y_and_err(tau, tau_err, Te, Te_err)

#             labels[key] = y
#             labels[key.replace("mean", "std")] = yerr
#         else:
#             idx = param_indices[key.split('_')[0] + '_' + key.split('_')[1]]
#             mean, std = calc_mean_and_std(samples, idx)
#             labels[key] = mean
#             labels[key.replace("mean", "std")] = std

#     print("Results:")
#     for key, value in labels.items():
#         if ("y_" in key) or ("tau_" in key):
#             # format y values to scientific notation
#             print(f"{key}: {value:.3e}")
#         else:
#             print(f"{key}: {value:.3f}")

#     # for each parameter that is common between labels and compton_y_fits, compare the values 
#     # and print the difference in standard deviations
#     print("\nComparison with Compton y fits:")
    
#     for key in labels.keys():
#         if "mean" not in key:
#             continue
#         if key in compton_y_fits:
#             diff = (labels[key] - compton_y_fits[key]) / (np.sqrt(labels[key.replace("mean", "std")]**2 + compton_y_fits[key.replace("mean", "std")]**2))
#             print(f"{key}: {labels[key]:.7f} vs {compton_y_fits[key]:.7f} | Diff: {diff:.2f} σ")
#         else:
#             print(f"{key} not found in Compton y fits")

#     a401_y_vector = get_y_vector(samples, param_indices['tau_a401'], param_indices['Te_a401'])
#     a399_y_vector = get_y_vector(samples, param_indices['tau_a399'], param_indices['Te_a399'])
#     fil_y_vector = get_y_vector(samples, param_indices['tau_fil'], param_indices['Te_fil'])

#     return a401_y_vector, a399_y_vector, fil_y_vector

#     # # Plot the y distributions
#     # plt.figure(figsize=(15, 8))
#     # plt.hist(a401_y_vector, bins=50, alpha=0.5, label='A401 y', density=True, histtype='stepfilled')
#     # plt.hist(a399_y_vector, bins=50, alpha=0.5, label='A399 y', density=True, histtype='stepfilled')
#     # plt.hist(fil_y_vector, bins=50, alpha=0.5, label='Filament y', density=True, histtype='stepfilled')

#     # # plot the compton y- y paramter assuming a normal distribution (np.normal(loc=mean, scale=std))
#     # plt.hist(np.random.normal(compton_y_fits['y_a401_mean'], compton_y_fits['y_a401_std'], 10000), 
#     #          histtype='stepfilled',
#     #          bins=50, alpha=0.5, label='A401 y (Compton fit)', density=True)
#     # plt.hist(np.random.normal(compton_y_fits['y_a399_mean'], compton_y_fits['y_a399_std'], 10000),
#     #             histtype='stepfilled',
#     #          bins=50, alpha=0.5, label='A399 y (Compton fit)', density=True)
#     # plt.hist(np.random.normal(compton_y_fits['y_fil_mean'], compton_y_fits['y_fil_std'], 10000),
#     #             histtype='stepfilled',
#     #          bins=50, alpha=0.5, label='Filament y (Compton fit)', density=True)
    
#     # plt.xlabel('Compton y')
#     # plt.ylabel('Density')
#     # plt.title('Compton y Distributions')
#     # plt.legend()
#     # plt.show()

In [7]:
def result_comparer_indv(samples, indv_all_fits, param_indices, cf_path):
    # create a dictionary of the sample results
    # create the labels first for the dictionary

    ## Take care of RA and DEC values 
    cf = ut.get_config_file(f'{cf_path}')

    region = ut.get_region(cf['region_center_ra'], 
                           cf['region_center_dec'], 
                           cf['region_width'])

    dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"
    data_ref = enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
                              box=region)
    
    a401_ra_pix = samples[:, param_indices['ra_a401']]
    a401_dec_pix = samples[:, param_indices['dec_a401']]

    a399_ra_pix = samples[:, param_indices['ra_a399']]
    a399_dec_pix = samples[:, param_indices['dec_a399']]

    fil_ra_pix = samples[:, param_indices['fil_ra']]
    fil_dec_pix = samples[:, param_indices['fil_dec']]

    ra_a401, dec_a401 = data_ref.wcs.celestial.wcs_pix2world(a401_ra_pix, a401_dec_pix, 0)
    samples[:, param_indices['ra_a401']] = ra_a401
    samples[:, param_indices['dec_a401']] = dec_a401

    ra_a399, dec_a399 = data_ref.wcs.celestial.wcs_pix2world(a399_ra_pix, a399_dec_pix, 0)
    samples[:, param_indices['ra_a399']] = ra_a399
    samples[:, param_indices['dec_a399']] = dec_a399

    fil_ra, fil_dec = data_ref.wcs.celestial.wcs_pix2world(fil_ra_pix, fil_dec_pix, 0)
    samples[:, param_indices['fil_ra']] = fil_ra
    samples[:, param_indices['fil_dec']] = fil_dec

    samples[:, param_indices["fil_l0"]] *= 0.5
    samples[:, param_indices["fil_w0"]] *= 0.5

    samples[:, param_indices["R_a401"]] = 1 / samples[:, param_indices["R_a401"]]
    samples[:, param_indices["R_a399"]] = 1 / samples[:, param_indices["R_a399"]]
    
    # create a dictionary to hold the results
    labels = {
        "ra_a401_mean": 0,
        "ra_a401_std": 0,

        "dec_a401_mean": 0,
        "dec_a401_std": 0,

        "beta_a401_mean": 0,
        "beta_a401_std": 0,

        "rc_a401_mean": 0,
        "rc_a401_std": 0,

        "R_a401_mean": 0,
        "R_a401_std": 0,

        "theta_a401_mean": 0,
        "theta_a401_std": 0,

        "tau_a401_mean": 0,
        "tau_a401_std": 0,

        "Te_a401_mean": 0,
        "Te_a401_std": 0,

        "AD_a401_mean": 0,
        "AD_a401_std": 0,

        "vr_a401_mean": 0,
        "vr_a401_std": 0,

        "ra_a399_mean": 0,
        "ra_a399_std": 0,

        "dec_a399_mean": 0,
        "dec_a399_std": 0,

        "beta_a399_mean": 0,
        "beta_a399_std": 0,

        "rc_a399_mean": 0,
        "rc_a399_std": 0,

        "R_a399_mean": 0,
        "R_a399_std": 0,

        "theta_a399_mean": 0,
        "theta_a399_std": 0,

        "tau_a399_mean": 0,
        "tau_a399_std": 0,

        "Te_a399_mean": 0,
        "Te_a399_std": 0,

        "AD_a399_mean": 0,
        "AD_a399_std": 0,

        "vr_a399_mean": 0,
        "vr_a399_std": 0,

        "fil_ra_mean": 0,
        "fil_ra_std": 0,

        "fil_dec_mean": 0,
        "fil_dec_std": 0,

        "fil_l0_mean": 0,
        "fil_l0_std": 0,

        "fil_w0_mean": 0,
        "fil_w0_std": 0,

        "tau_fil_mean": 0,
        "tau_fil_std": 0,

        "Te_fil_mean": 0,
        "Te_fil_std": 0,

        "fil_AD_mean": 0,
        "fil_AD_std": 0,

        "fil_vavg_mean": 0,
        "fil_vavg_std": 0,

        "y_a401_mean": 0,
        "y_a401_std": 0,
        
        "y_a399_mean": 0,
        "y_a399_std": 0,

        "y_fil_mean": 0,
        "y_fil_std": 0,
    }

    # calculate the mean and std for each param in labels
    for key in labels.keys():
        if "std" in key:
            continue
        
        if "y_" in key:
            object_name = key.split('_')[1]
            Te = labels[f'Te_{object_name}_mean']
            Te_err = labels[f'Te_{object_name}_std']
            tau = labels[f'tau_{object_name}_mean']
            tau_err = labels[f'tau_{object_name}_std']
            y, yerr = calc_y_and_err(tau, tau_err, Te, Te_err)

            labels[key] = y
            labels[key.replace("mean", "std")] = yerr
        else:
            idx = param_indices[key.split('_')[0] + '_' + key.split('_')[1]]
            mean, std = calc_mean_and_std(samples, idx)
            labels[key] = mean
            labels[key.replace("mean", "std")] = std

    print("Results:")
    for key, value in labels.items():
        if ("y_" in key) or ("tau_" in key):
            # format y values to scientific notation
            print(f"{key}: {value:.3e}")
        else:
            print(f"{key}: {value:.3f}")

    # for each parameter that is common between labels and compton_y_fits, compare the values 
    # and print the difference in standard deviations
    print("\nComparison with ind fits at baseline:")
    
    for key in labels.keys():
        if "mean" not in key:
            continue
        if ("tau_" in key) or ("Te_" in key) or ("vavg" in key) or ("vr_" in key) or ("y_" in key):
        #if ("vavg" in key) or ("vr_" in key):
            if key in indv_all_fits:
                diff_percent = ((labels[key] - indv_all_fits[key]) / indv_all_fits[key]) * 100
                print(f"{key}: {labels[key]:.7f} vs {indv_all_fits[key]:.7f} | Diff: {diff_percent:.2f}%")
            else:
                print(f"{key} not found in indv fits")

    #a401_y_vector = get_y_vector(samples, param_indices['tau_a401'], param_indices['Te_a401'])
    #a399_y_vector = get_y_vector(samples, param_indices['tau_a399'], param_indices['Te_a399'])
    #fil_y_vector = get_y_vector(samples, param_indices['tau_fil'], param_indices['Te_fil'])

    #return a401_y_vector, a399_y_vector, fil_y_vector


In [8]:
configs = [
    "/home/gill/research/ACT/multi-freq-bridge/configs/case23_ajay.yaml", 
    "/home/gill/research/ACT/multi-freq-bridge/configs/case34_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case35_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case36_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case37_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case38_ajay.yaml"
]

# indv_all_fits = {
#     # Abell 401 values from Case 23 baseline results
#     "ra_a401_mean": 44.741,
#     "ra_a401_std": 0.002,
#     "dec_a401_mean": 13.580,
#     "dec_a401_std": 0.002,
#     "beta_a401_mean": 1.268,
#     "beta_a401_std": 0.091,
#     "rc_a401_mean": 4.923,
#     "rc_a401_std": 0.525,
#     "R_a401_mean": 0.800,
#     "R_a401_std": 0.055,
#     "theta_a401_mean": 107.439,
#     "theta_a401_std": 6.635,
#     "y_a401_mean": 1.194e-04,
#     "y_a401_std": 8.678e-06,
#     "Te_a401_mean": 8.459,
#     "Te_a401_std": 0.248,
#     "tau_a401_mean": 7.211e-03,
#     "tau_a401_std": 4.796e-04,
#     "a401_AD_mean": 237849.943,
#     "a401_AD_std": 198914.107,

#     # Abell 399 values from Case 23 baseline results
#     "ra_a399_mean": 44.465,
#     "ra_a399_std": 0.003,
#     "dec_a399_mean": 13.040,
#     "dec_a399_std": 0.004,
#     "beta_a399_mean": 1.018,
#     "beta_a399_std": 0.069,
#     "rc_a399_mean": 4.002,
#     "rc_a399_std": 0.539,
#     "R_a399_mean": 0.909,
#     "R_a399_std": 0.091,
#     "theta_a399_mean": 109.751,
#     "theta_a399_std": 27.204,
#     "y_a399_mean": 9.201e-05,
#     "y_a399_std": 7.004e-06,
#     "Te_a399_mean": 7.228,
#     "Te_a399_std": 0.186,
#     "tau_a399_mean": 6.505e-03,
#     "tau_a399_std": 4.659e-04,
#     "a399_AD_mean": 146232.378,
#     "a399_AD_std": 148747.819,

#     # Filament values from Case 23 baseline results
#     "fil_ra_mean": 44.669,
#     "fil_ra_std": 0.021,
#     "fil_dec_mean": 13.342,
#     "fil_dec_std": 0.031,
#     "fil_l0_mean": 15.523,
#     "fil_l0_std": 2.069,
#     "fil_w0_mean": 12.386,
#     "fil_w0_std": 1.127,
#     "y_fil_mean": 1.460e-05,
#     "y_fil_std": 2.159e-06,
#     "Te_fil_mean": 6.507,
#     "Te_fil_std": 0.350,
#     "tau_fil_mean": 1.146e-03,
#     "tau_fil_std": 1.579e-04,
#     "fil_AD_mean": 76180.688,
#     "fil_AD_std": 76570.746,

#     # Velocity parameters from Case 23 baseline results
#     "vr_a401_mean": -1072.985,
#     "vr_a401_std": 712.037,
#     "vr_a399_mean": 644.949,
#     "vr_a399_std": 683.569,
#     "fil_vavg_mean": 770.786,
#     "fil_vavg_std": 1559.042,
# }

print("Case 23: baseline (X-ray)")
result_comparer_indv(samples_case23, indv_all_fits, param_indices, configs[0])

print("\nCase 34: 5% higher than X-ray")
result_comparer_indv(samples_case34, indv_all_fits, param_indices, configs[1])

print("\nCase 35: 10% higher than X-ray")
result_comparer_indv(samples_case35, indv_all_fits, param_indices, configs[2])

print("\nCase 36: 15% higher than X-ray")
result_comparer_indv(samples_case36, indv_all_fits, param_indices, configs[3])

print("\nCase 37: 20% higher than X-ray")
result_comparer_indv(samples_case37, indv_all_fits, param_indices, configs[4])

print("\nCase 38: 25% higher than X-ray")
result_comparer_indv(samples_case38, indv_all_fits, param_indices, configs[5])

Case 23: baseline (X-ray)
Results:
ra_a401_mean: 44.741
ra_a401_std: 0.002
dec_a401_mean: 13.580
dec_a401_std: 0.002
beta_a401_mean: 1.254
beta_a401_std: 0.091
rc_a401_mean: 4.971
rc_a401_std: 0.525
R_a401_mean: 0.795
R_a401_std: 0.055
theta_a401_mean: 107.377
theta_a401_std: 6.635
tau_a401_mean: 7.145e-03
tau_a401_std: 4.796e-04
Te_a401_mean: 8.414
Te_a401_std: 0.248
AD_a401_mean: 80834.653
AD_a401_std: 198914.107
vr_a401_mean: -1152.993
vr_a401_std: 712.037
ra_a399_mean: 44.465
ra_a399_std: 0.003
dec_a399_mean: 13.040
dec_a399_std: 0.004
beta_a399_mean: 1.000
beta_a399_std: 0.069
rc_a399_mean: 3.998
rc_a399_std: 0.539
R_a399_mean: 0.884
R_a399_std: 0.091
theta_a399_mean: 107.330
theta_a399_std: 27.204
tau_a399_mean: 6.432e-03
tau_a399_std: 4.659e-04
Te_a399_mean: 7.240
Te_a399_std: 0.186
AD_a399_mean: 1087.053
AD_a399_std: 148747.819
vr_a399_mean: 780.433
vr_a399_std: 683.569
fil_ra_mean: 44.670
fil_ra_std: 0.021
fil_dec_mean: 13.338
fil_dec_std: 0.031
fil_l0_mean: 15.191
fil_l0_std:

In [ ]:


configs = [
    "/home/gill/research/ACT/multi-freq-bridge/configs/case23_ajay.yaml", 
    "/home/gill/research/ACT/multi-freq-bridge/configs/case34_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case35_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case36_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case37_ajay.yaml",
    "/home/gill/research/ACT/multi-freq-bridge/configs/case38_ajay.yaml",
]

a401_y_vector_case23, a399_y_vector_case23, fil_y_vector_case23 = result_comparer(samples_case23, compton_y_fits, param_indices, configs[0])


In [ ]:
# plot the velocity distributions for each case for A401, A399 and Filament
fig, axs = plt.subplots(3, 2, figsize=(15, 15))

# Get baseline means for comparison using calc_mean_and_std
baseline_vr_a401_mean, _ = calc_mean_and_std(cases[0][0], param_indices['vr_a401'])
baseline_vr_a399_mean, _ = calc_mean_and_std(cases[0][0], param_indices['vr_a399'])
baseline_fil_vavg_mean, _ = calc_mean_and_std(cases[0][0], param_indices['fil_vavg'])

# A401 velocity distributions
for i, (samples, case_name) in enumerate(cases):
    vr_a401 = samples[:, param_indices['vr_a401']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['vr_a401'])
    percent_diff = ((mean_val - baseline_vr_a401_mean) / baseline_vr_a401_mean) * 100
    axs[0, 0].hist(vr_a401, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.0f}, sigma={std_val:.0f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 0].set_title('A401 Velocity Distribution')
axs[0, 0].set_xlabel('Velocity (km/s)')
axs[0, 0].set_ylabel('Density')
axs[0, 0].legend(fontsize=10)

# A399 velocity distributions
for i, (samples, case_name) in enumerate(cases):
    vr_a399 = samples[:, param_indices['vr_a399']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['vr_a399'])
    percent_diff = ((mean_val - baseline_vr_a399_mean) / baseline_vr_a399_mean) * 100
    axs[0, 1].hist(vr_a399, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.0f}, sigma={std_val:.0f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 1].set_title('A399 Velocity Distribution')
axs[0, 1].set_xlabel('Velocity (km/s)')
axs[0, 1].set_ylabel('Density')
axs[0, 1].legend(fontsize=10)

# Filament velocity distributions
for i, (samples, case_name) in enumerate(cases):
    fil_vavg = samples[:, param_indices['fil_vavg']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['fil_vavg'])
    percent_diff = ((mean_val - baseline_fil_vavg_mean) / baseline_fil_vavg_mean) * 100
    axs[1, 0].hist(fil_vavg, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.0f}, sigma={std_val:.0f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[1, 0].set_title('Filament Velocity Distribution')
axs[1, 0].set_xlabel('Velocity (km/s)')
axs[1, 0].set_ylabel('Density')
axs[1, 0].legend(fontsize=10)

# Remove unused subplots
axs[1, 1].axis('off')
axs[2, 0].axis('off')
axs[2, 1].axis('off')

plt.tight_layout()
plt.show()

# Temperature distributions for A401, A399 and Filament
fig, axs = plt.subplots(3, 2, figsize=(15, 15))

# Get baseline means for comparison using calc_mean_and_std
baseline_Te_a401_mean, _ = calc_mean_and_std(cases[0][0], param_indices['Te_a401'])
baseline_Te_a399_mean, _ = calc_mean_and_std(cases[0][0], param_indices['Te_a399'])
baseline_Te_fil_mean, _ = calc_mean_and_std(cases[0][0], param_indices['Te_fil'])

# A401 temperature distributions
for i, (samples, case_name) in enumerate(cases):
    Te_a401 = samples[:, param_indices['Te_a401']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['Te_a401'])
    percent_diff = ((mean_val - baseline_Te_a401_mean) / baseline_Te_a401_mean) * 100
    axs[0, 0].hist(Te_a401, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.2f}, sigma={std_val:.2f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 0].set_title('A401 Temperature Distribution')
axs[0, 0].set_xlabel('Temperature (keV)')
axs[0, 0].set_ylabel('Density')
axs[0, 0].legend(fontsize=10)

# A399 temperature distributions
for i, (samples, case_name) in enumerate(cases):
    Te_a399 = samples[:, param_indices['Te_a399']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['Te_a399'])
    percent_diff = ((mean_val - baseline_Te_a399_mean) / baseline_Te_a399_mean) * 100
    axs[0, 1].hist(Te_a399, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.2f}, sigma={std_val:.2f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 1].set_title('A399 Temperature Distribution')
axs[0, 1].set_xlabel('Temperature (keV)')
axs[0, 1].set_ylabel('Density')
axs[0, 1].legend(fontsize=10)

# Filament temperature distributions
for i, (samples, case_name) in enumerate(cases):
    Te_fil = samples[:, param_indices['Te_fil']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['Te_fil'])
    percent_diff = ((mean_val - baseline_Te_fil_mean) / baseline_Te_fil_mean) * 100
    axs[1, 0].hist(Te_fil, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.2f}, sigma={std_val:.2f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[1, 0].set_title('Filament Temperature Distribution')
axs[1, 0].set_xlabel('Temperature (keV)')
axs[1, 0].set_ylabel('Density')
axs[1, 0].legend(fontsize=10)

# Remove unused subplot
axs[1, 1].axis('off')
axs[2, 0].axis('off')
axs[2, 1].axis('off')

plt.tight_layout()
plt.show()

# Optical depth (tau) distributions for A401, A399 and Filament
fig, axs = plt.subplots(3, 2, figsize=(15, 15))

# Get baseline means for comparison
baseline_tau_a401_mean, _ = calc_mean_and_std(cases[0][0], param_indices['tau_a401'])
baseline_tau_a399_mean, _ = calc_mean_and_std(cases[0][0], param_indices['tau_a399'])
baseline_tau_fil_mean, _ = calc_mean_and_std(cases[0][0], param_indices['tau_fil'])

# A401 tau distributions
for i, (samples, case_name) in enumerate(cases):
    tau_a401 = samples[:, param_indices['tau_a401']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['tau_a401'])
    percent_diff = ((mean_val - baseline_tau_a401_mean) / baseline_tau_a401_mean) * 100
    axs[0, 0].hist(tau_a401, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.4f}, sigma={std_val:.4f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 0].set_title('A401 Optical Depth Distribution')
axs[0, 0].set_xlabel('Optical Depth')
axs[0, 0].set_ylabel('Density')
axs[0, 0].legend(fontsize=10)

# A399 tau distributions
for i, (samples, case_name) in enumerate(cases):
    tau_a399 = samples[:, param_indices['tau_a399']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['tau_a399'])
    percent_diff = ((mean_val - baseline_tau_a399_mean) / baseline_tau_a399_mean) * 100
    axs[0, 1].hist(tau_a399, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.4f}, sigma={std_val:.4f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 1].set_title('A399 Optical Depth Distribution')
axs[0, 1].set_xlabel('Optical Depth')
axs[0, 1].set_ylabel('Density')
axs[0, 1].legend(fontsize=10)

# Filament tau distributions
for i, (samples, case_name) in enumerate(cases):
    tau_fil = samples[:, param_indices['tau_fil']]
    mean_val, std_val = calc_mean_and_std(samples, param_indices['tau_fil'])
    percent_diff = ((mean_val - baseline_tau_fil_mean) / baseline_tau_fil_mean) * 100
    axs[1, 0].hist(tau_fil, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.4f}, sigma={std_val:.4f} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[1, 0].set_title('Filament Optical Depth Distribution')
axs[1, 0].set_xlabel('Optical Depth')
axs[1, 0].set_ylabel('Density')
axs[1, 0].legend(fontsize=10)

# Remove unused subplot
axs[1, 1].axis('off')
axs[2, 0].axis('off')
axs[2, 1].axis('off')

plt.tight_layout()
plt.show()

# Compton-y distributions for A401, A399 and Filament
fig, axs = plt.subplots(3, 2, figsize=(15, 15))

# Get y vectors for all cases
a401_y_vectors = []
a399_y_vectors = []
fil_y_vectors = []

for i, (samples, case_name) in enumerate(cases):
    a401_y_vectors.append(get_y_vector(samples, param_indices['tau_a401'], param_indices['Te_a401']))
    a399_y_vectors.append(get_y_vector(samples, param_indices['tau_a399'], param_indices['Te_a399']))
    fil_y_vectors.append(get_y_vector(samples, param_indices['tau_fil'], param_indices['Te_fil']))

# Get baseline means for comparison
baseline_y_a401_mean = np.mean(a401_y_vectors[0])
baseline_y_a399_mean = np.mean(a399_y_vectors[0])
baseline_y_fil_mean = np.mean(fil_y_vectors[0])

# A401 Compton-y distributions
for i in range(len(cases)):
    y_a401 = a401_y_vectors[i]
    mean_val = np.mean(y_a401)
    std_val = np.std(y_a401)
    percent_diff = ((mean_val - baseline_y_a401_mean) / baseline_y_a401_mean) * 100
    axs[0, 0].hist(y_a401, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.2e}, sigma={std_val:.2e} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 0].set_title('A401 Compton-y Distribution')
axs[0, 0].set_xlabel('Compton-y')
axs[0, 0].set_ylabel('Density')
axs[0, 0].legend(fontsize=10)

# A399 Compton-y distributions
for i in range(len(cases)):
    y_a399 = a399_y_vectors[i]
    mean_val = np.mean(y_a399)
    std_val = np.std(y_a399)
    percent_diff = ((mean_val - baseline_y_a399_mean) / baseline_y_a399_mean) * 100
    axs[0, 1].hist(y_a399, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.2e}, sigma={std_val:.2e} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[0, 1].set_title('A399 Compton-y Distribution')
axs[0, 1].set_xlabel('Compton-y')
axs[0, 1].set_ylabel('Density')
axs[0, 1].legend(fontsize=10)

# Filament Compton-y distributions
for i in range(len(cases)):
    y_fil = fil_y_vectors[i]
    mean_val = np.mean(y_fil)
    std_val = np.std(y_fil)
    percent_diff = ((mean_val - baseline_y_fil_mean) / baseline_y_fil_mean) * 100
    axs[1, 0].hist(y_fil, bins=50, alpha=1, 
                   label=f'{case_labels[i]}: mu={mean_val:.2e}, sigma={std_val:.2e} ({percent_diff:.1f}%)', 
                   density=True, histtype='step', lw=2)
axs[1, 0].set_title('Filament Compton-y Distribution')
axs[1, 0].set_xlabel('Compton-y')
axs[1, 0].set_ylabel('Density')
axs[1, 0].legend(fontsize=10)

# Remove unused subplot
axs[1, 1].axis('off')
axs[2, 0].axis('off')
axs[2, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# plot the y distributions for each case
# plot the y distributions for each case using subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# A401 Compton-y distributions
axes[0].hist(a401_y_vector_case23, bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=2)
axes[0].hist(a401_y_vector_case34, bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=2)
axes[0].hist(a401_y_vector_case35, bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=2)
axes[0].hist(a401_y_vector_case36, bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=2)
axes[0].hist(a401_y_vector_case37, bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=2)
axes[0].hist(a401_y_vector_case38, bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=2)
axes[0].hist(np.random.normal(compton_y_fits['y_a401_mean'], compton_y_fits['y_a401_std'], 100000), 
             histtype='step', color='black', linestyle='-',
             bins=100, alpha=1, label='$y$-map (Hincks et al. 2022)', density=True, lw=2)
axes[0].set_xlabel('Compton-$y$ (Abell 401)')
axes[0].legend(fontsize=10)
axes[0].set_yticks([])
axes[0].set_xlim(0.00009, 0.00015)

# A399 Compton-y distributions
axes[1].hist(a399_y_vector_case23, bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=2)
axes[1].hist(a399_y_vector_case34, bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=2)
axes[1].hist(a399_y_vector_case35, bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=2)
axes[1].hist(a399_y_vector_case36, bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=2)
axes[1].hist(a399_y_vector_case37, bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=2)
axes[1].hist(a399_y_vector_case38, bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=2)
axes[1].hist(np.random.normal(compton_y_fits['y_a399_mean'], compton_y_fits['y_a399_std'], 100000), 
             histtype='step', 
             bins=100, alpha=1, label='$y$-map', density=True, lw=2, color='black', linestyle='-')
axes[1].set_xlabel('Compton-$y$ (Abell 399)')
# axes[1].legend(fontsize=10)
axes[1].set_yticks([])
axes[1].set_xlim(0.00006, 0.00011)

# Filament Compton-y distributions
axes[2].hist(fil_y_vector_case23, bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=2)
axes[2].hist(fil_y_vector_case34, bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=2)
axes[2].hist(fil_y_vector_case35, bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=2)
axes[2].hist(fil_y_vector_case36, bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=2)
axes[2].hist(fil_y_vector_case37, bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=2)
axes[2].hist(fil_y_vector_case38, bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=2)
axes[2].hist(np.random.normal(compton_y_fits['y_fil_mean'], compton_y_fits['y_fil_std'], 100000), 
             histtype='step', 
             bins=100, alpha=1, label='$y$-map', density=True, lw=2, color='black', linestyle='-')
axes[2].set_xlabel('Compton-$y$ (Filament)')
# axes[2].legend(fontsize=10)
axes[2].set_yticks([])
axes[2].set_xlim(0.000003, 0.000025)

# plt.tight_layout()
plt.show()


In [ ]:
# Te distributions for each case
plt.figure(figsize=(12, 5))
plt.hist(samples_case10[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=3)
plt.hist(samples_case24[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=3)
plt.hist(samples_case25[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=3)
plt.hist(samples_case26[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=3)
plt.hist(samples_case27[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=3)
plt.hist(samples_case28[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=3)
plt.hist(samples_case29[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[6], density=True, histtype='step', lw=3)
plt.hist(samples_case30[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[7], density=True, histtype='step', lw=3)
plt.hist(samples_case31[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[8], density=True, histtype='step', lw=3)
plt.hist(samples_case32[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[9], density=True, histtype='step', lw=3)
plt.hist(samples_case33[:, param_indices['Te_a401']], bins=200, alpha=1, label=case_labels[10], density=True, histtype='step', lw=3)
plt.xlabel('Temperature (keV)')
plt.legend(fontsize=12)
plt.gca().set_yticks([])
#plt.savefig("temperature_a401_distributions.png", bbox_inches='tight')
plt.show()
# Te distributions for A399
plt.figure(figsize=(12, 5))
plt.hist(samples_case10[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=3)
plt.hist(samples_case24[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=3)
plt.hist(samples_case25[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=3)
plt.hist(samples_case26[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=3)     
plt.hist(samples_case27[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=3)
plt.hist(samples_case28[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=3)
plt.hist(samples_case29[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[6], density=True, histtype='step', lw=3)
plt.hist(samples_case30[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[7], density=True, histtype='step', lw=3)
plt.hist(samples_case31[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[8], density=True, histtype='step', lw=3)
plt.hist(samples_case32[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[9], density=True, histtype='step', lw=3)
plt.hist(samples_case33[:, param_indices['Te_a399']], bins=200, alpha=1, label=case_labels[10], density=True, histtype='step', lw=3)
plt.xlabel('Temperature (keV)')
plt.legend(fontsize=12)
plt.gca().set_yticks([])
#plt.savefig("temperature_a399_distributions.png", bbox_inches='tight')
plt.show()
# Te distributions for Filament
plt.figure(figsize=(12, 5))
plt.hist(samples_case10[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=3)  
plt.hist(samples_case24[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=3)
plt.hist(samples_case25[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=3)
plt.hist(samples_case26[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=3)
plt.hist(samples_case27[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=3)
plt.hist(samples_case28[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=3)
plt.hist(samples_case29[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[6], density=True, histtype='step', lw=3)
plt.hist(samples_case30[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[7], density=True, histtype='step', lw=3)
plt.hist(samples_case31[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[8], density=True, histtype='step', lw=3)
plt.hist(samples_case32[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[9], density=True, histtype='step', lw=3)
plt.hist(samples_case33[:, param_indices['Te_fil']], bins=200, alpha=1, label=case_labels[10], density=True, histtype='step', lw=3)
plt.xlabel('Temperature (keV)')
plt.legend(fontsize=12)
plt.gca().set_yticks([])
#plt.savefig("temperature_filament_distributions.png", bbox_inches='tight')
plt.show()

In [ ]:
# tau histograms for a401   
plt.figure(figsize=(12, 5))
plt.hist(samples_case10[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=3)
#plt.hist(samples_case24[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=3)
#plt.hist(samples_case25[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=3)
# plt.hist(samples_case26[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=3)
plt.hist(samples_case27[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=3)
plt.hist(samples_case28[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=3)
plt.hist(samples_case29[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[6], density=True, histtype='step', lw=3)
plt.hist(samples_case30[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[7], density=True, histtype='step', lw=3)
plt.hist(samples_case31[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[8], density=True, histtype='step', lw=3)    
plt.hist(samples_case32[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[9], density=True, histtype='step', lw=3)
plt.hist(samples_case33[:, param_indices['tau_a401']], bins=200, alpha=1, label=case_labels[10], density=True, histtype='step', lw=3)
plt.xlabel('Optical depth (A401)')
plt.legend(fontsize=12)
plt.gca().set_yticks([])
plt.xlim(0.004, 0.01)
#plt.savefig("tau_a401_distributions.png", bbox_inches='tight')
plt.show()

# tau histograms for a399
plt.figure(figsize=(12, 5))
plt.hist(samples_case10[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=3)
#plt.hist(samples_case24[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=3)
#plt.hist(samples_case25[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=3)    
# plt.hist(samples_case26[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=3)
plt.hist(samples_case27[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=3)
plt.hist(samples_case28[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=3)
plt.hist(samples_case29[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[6], density=True, histtype='step', lw=3)
plt.hist(samples_case30[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[7], density=True, histtype='step', lw=3)
plt.hist(samples_case31[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[8], density=True, histtype='step', lw=3)
plt.hist(samples_case32[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[9], density=True, histtype='step', lw=3)
plt.hist(samples_case33[:, param_indices['tau_a399']], bins=200, alpha=1, label=case_labels[10], density=True, histtype='step', lw=3)
plt.xlabel('Optical depth (A399)')
plt.legend(fontsize=12)
plt.gca().set_yticks([])
plt.xlim(0.003, 0.008)
#plt.savefig("tau_a399_distributions.png", bbox_inches='tight')
plt.show()

# tau histograms for filament
plt.figure(figsize=(12, 5))
plt.hist(samples_case10[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[0], density=True, histtype='step', lw=3)
#plt.hist(samples_case24[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[1], density=True, histtype='step', lw=3)
#plt.hist(samples_case25[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[2], density=True, histtype='step', lw=3)
# plt.hist(samples_case26[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[3], density=True, histtype='step', lw=3)
plt.hist(samples_case27[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[4], density=True, histtype='step', lw=3)
plt.hist(samples_case28[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[5], density=True, histtype='step', lw=3)
plt.hist(samples_case29[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[6], density=True, histtype='step', lw=3)
plt.hist(samples_case30[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[7], density=True, histtype='step', lw=3)
plt.hist(samples_case31[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[8], density=True, histtype='step', lw=3)
plt.hist(samples_case32[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[9], density=True, histtype='step', lw=3)
plt.hist(samples_case33[:, param_indices['tau_fil']], bins=200, alpha=1, label=case_labels[10], density=True, histtype='step', lw=3)
plt.xlabel('Optical depth (Filament)')
plt.legend(fontsize=12)
plt.gca().set_yticks([])
plt.xlim(0.0005, 0.0017)
#plt.savefig("tau_filament_distributions.png", bbox_inches='tight')
plt.show()

In [ ]:
# plot the histograms of the temperature for each case with three subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 15))

# A401 temperatures
for i, (samples, case_name) in enumerate(cases):
    Te_a401 = samples[:, param_indices['Te_a401']]
    axes[0].hist(Te_a401, bins=50, alpha=0.7, label=f'{case_name}', density=True, histtype='step')
axes[0].set_xlabel('Temperature (keV)')
axes[0].set_ylabel('Density')
axes[0].set_title('A401 Temperature Distributions')
axes[0].legend()

# A399 temperatures
for i, (samples, case_name) in enumerate(cases):
    Te_a399 = samples[:, param_indices['Te_a399']]
    axes[1].hist(Te_a399, bins=50, alpha=0.7, label=f'{case_name}', density=True, histtype='step')
axes[1].set_xlabel('Temperature (keV)')
axes[1].set_ylabel('Density')
axes[1].set_title('A399 Temperature Distributions')
axes[1].legend()

# Filament temperatures
for i, (samples, case_name) in enumerate(cases):
    Te_fil = samples[:, param_indices['Te_fil']]
    axes[2].hist(Te_fil, bins=50, alpha=1, lw=3, label=f'{case_name}', density=True, histtype='step')
axes[2].set_xlabel('Temperature (keV)')
axes[2].set_ylabel('Density')
axes[2].set_title('Filament Temperature Distributions')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
cf = ut.get_config_file('/home/gill/research/ACT/multi-freq-bridge/configs/case10_ajay.yaml')
region = ut.get_region(cf['region_center_ra'], 
                        cf['region_center_dec'], 
                        cf['region_width'])

dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"
data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
                            box=region))

plate_scale = 0.5
r500_a401 = plate_scale * ut.get_r500(mass=cf['c1_mass'], z=cf['c1_z']) * u.arcmin
r500_a399 = plate_scale * ut.get_r500(mass=cf['c2_mass'], z=cf['c2_z']) * u.arcmin

r_core_a401 = 4.5 * u.arcmin
r_core_a399 = 4.2 * u.arcmin

ra_center_a401 = cf["c1_ra"]
dec_center_a401 = cf["c1_dec"]
ra_center_a399 = cf["c2_ra"]
dec_center_a399 = cf["c2_dec"]

In [ ]:
from pixell import colorize
colorize.mpl_register("planck")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pixell import enmap, reproject
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS

# latex font
plt.rc('text', usetex=True)
plt.rc('font', family='sans-serif', size=22)

# ---- Read and Prepare Map ----
cf = ut.get_config_file('/home/gill/research/ACT/multi-freq-bridge/configs/case10_ajay.yaml')
region = ut.get_region(cf['region_center_ra'], cf['region_center_dec'], 1.*cf['region_width'])
dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"

data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", box=region))

# ---- Physical Parameters ----
plate_scale = 0.5  # arcmin/pixel

r500_a401 = plate_scale * ut.get_r500(mass=cf['c1_mass'], z=cf['c1_z']) * u.arcmin
r500_a399 = plate_scale * ut.get_r500(mass=cf['c2_mass'], z=cf['c2_z']) * u.arcmin

r_core_a401 = 4.5 * u.arcmin
r_core_a399 = 4.2 * u.arcmin

ra_center_a401 = cf["c1_ra"]
dec_center_a401 = cf["c1_dec"]
ra_center_a399 = cf["c2_ra"]
dec_center_a399 = cf["c2_dec"]

# ---- Setup WCS and Plot ----
wcs = data_ref.wcs
shape = data_ref.shape

fig = plt.figure(figsize=(10, 6))
ax = plt.subplot(projection=(wcs))

im = ax.imshow(data_ref, origin='lower', cmap='planck')
ax.invert_xaxis()  # Invert x-axis for RA
plt.colorbar(im, ax=ax, orientation='vertical', label='$\mu$K')

def draw_circle(ax, ra, dec, radius, **kwargs):
    """Draw a circle at (ra, dec) with radius in arcmin."""
    center = SkyCoord(ra*u.deg, dec*u.deg)
    theta = np.linspace(0, 2*np.pi, 200)
    offset = radius.to(u.deg).value
    ras = center.ra.deg + offset * np.cos(theta) / np.cos(np.deg2rad(center.dec.deg))
    decs = center.dec.deg + offset * np.sin(theta)
    ax.plot(ras, decs, transform=ax.get_transform('world'), **kwargs)

# ---- Draw circles ----
draw_circle(ax, ra_center_a401, dec_center_a401, r500_a401, color='black', lw=2, ls='-',label='$R_{500c}$ (A401)')
draw_circle(ax, ra_center_a401, dec_center_a401, r_core_a401, color='black', lw=1, ls='--', label=r'$R_{\rm core}$ (A401)')
draw_circle(ax, ra_center_a399, dec_center_a399, r500_a399, color='red', lw=2, ls='-',label='$R_{500c}$ (A399)')
draw_circle(ax, ra_center_a399, dec_center_a399, r_core_a399, color='red', lw=1, ls='--', label=r'$R_{\rm core}$ (A399)')

# ---- Plot Formatting ----
ax.set_xlabel("Right Ascension")
ax.set_ylabel("Declination")
ax.legend(loc='upper right', fontsize=14, facecolor='white', edgecolor='black', framealpha=1.0)

# add a text box with white background towards the bottom center of the plot saying "ACT PA5 f90"
textstr = 'ACT PA5 (90 GHz)'
props = dict(boxstyle='round', facecolor='white', alpha=1)
ax.text(0.5, 0.05, textstr, transform=ax.transAxes, fontsize=15,
        verticalalignment='center', horizontalalignment='center', bbox=props)
plt.tight_layout()
plt.savefig("../plots/map_r500_core.pdf", dpi=300, bbox_inches='tight')
plt.show()
